# 05. 세그먼트별 심층 비교 (Step 5)

3개 세그먼트 간 업종 구성비·연령 구성비·월별 변동성(CV)·건당 결제금액을 통계적으로 비교하고,
§3-3에서 언급된 "50대·60대 이상 비중이 낮지 않다"는 특이 시그널을 별도로 조명한다.
마지막으로 이상치 지역 케이스 스터디 2곳을 전수 데이터로 재확인한다.

In [1]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
from scipy import stats
from feature_engineering import BUZ_LIST, AGE_LABEL_LIST

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 160)

segment_df = pd.read_csv('../results/segment_classification.csv')
foreign_df = pd.read_csv('../data/processed/foreign_consumption_clean.csv', dtype={'STRD_YYMM': str})

# 시각화·통계용으로 업종/연령 비중 피처를 다시 결합 (03번 산출물에는 핵심 지수만 저장했으므로)
from feature_engineering import build_region_features
features_full = build_region_features(foreign_df)
merged = segment_df[['SIDO_NM', 'CCG_NM', 'segment']].merge(features_full, on=['SIDO_NM', 'CCG_NM'])
SEGMENT_ORDER = ['생활밀착형', '로컬미식형', '프리미엄외식형']


## 5-1. 업종 구성비 비교 (Kruskal-Wallis)

세그먼트(3개) 간 각 업종 비중 분포에 유의한 차이가 있는지 비모수 분산분석(Kruskal-Wallis)으로 검정한다.

In [2]:
rows = []
for buz in BUZ_LIST:
    col = f'buz_pct_{buz}'
    samples = [g[col].values for _, g in merged.groupby('segment')]
    h, p = stats.kruskal(*samples)
    means = merged.groupby('segment')[col].mean().reindex(SEGMENT_ORDER)
    rows.append({'업종': buz, **{f'{s}_평균(%)': round(means[s], 1) for s in SEGMENT_ORDER},
                 'H통계량': round(h, 1), 'p_value': p, '유의(p<0.05)': p < 0.05})
buz_stat = pd.DataFrame(rows).sort_values('p_value')
buz_stat


,업종,생활밀착형_평균(%),로컬미식형_평균(%),프리미엄외식형_평균(%),H통계량,p_value,유의(p<0.05)
8,슈퍼마켓,38.1,22.0,16.6,118.4,1.939605e-26,True
1,일반한식,21.1,32.1,27.0,94.1,3.619208e-21,True
0,서양음식,7.2,8.7,15.6,89.6,3.525280e-20,True
9,제과점,1.5,2.3,2.6,76.1,2.958869e-17,True
3,일식회집,0.5,1.3,2.2,57.9,2.663625e-13,True
2,중국음식,1.9,3.6,3.5,52.6,3.768419e-12,True
10,스넥,2.3,3.1,3.2,41.7,8.908748e-10,True
6,편의점,21.8,18.8,22.8,30.1,2.905046e-07,True
7,대형할인점,5.6,8.0,6.5,4.5,1.035839e-01,False
5,갈비전문점,0.0,0.0,0.0,2.8,2.470100e-01,False


## 5-2. 연령대 구성비 비교 (Kruskal-Wallis)

In [3]:
rows = []
for age in AGE_LABEL_LIST:
    col = f'age_pct_{age}'
    samples = [g[col].values for _, g in merged.groupby('segment')]
    h, p = stats.kruskal(*samples)
    means = merged.groupby('segment')[col].mean().reindex(SEGMENT_ORDER)
    rows.append({'연령대': age, **{f'{s}_평균(%)': round(means[s], 1) for s in SEGMENT_ORDER},
                 'H통계량': round(h, 1), 'p_value': p, '유의(p<0.05)': p < 0.05})
age_stat = pd.DataFrame(rows)
age_stat


,연령대,생활밀착형_평균(%),로컬미식형_평균(%),프리미엄외식형_평균(%),H통계량,p_value,유의(p<0.05)
0,20대 이하,1.3,1.0,4.2,88.2,7.120619e-20,True
1,20대,22.6,12.7,27.7,174.6,1.197060e-38,True
2,30대,30.6,19.7,20.1,137.5,1.365362e-30,True
3,40대,19.0,21.2,16.6,61.0,5.689324e-14,True
4,50대,14.9,23.1,16.5,169.5,1.585252e-37,True
5,60대 이상,11.7,22.2,15.0,139.3,5.622848e-31,True


## 5-3. 월별 변동성(CV) 및 건당 결제금액 비교

In [4]:
for col, label in [('월별변동계수CV', '월별 변동계수(CV)'), ('건당평균결제금액', '건당 평균결제금액')]:
    samples = [g[col].values for _, g in segment_df.groupby('segment')]
    h, p = stats.kruskal(*samples)
    means = segment_df.groupby('segment')[col].mean().reindex(SEGMENT_ORDER)
    print(f"[{label}] 세그먼트별 평균: {means.round(2).to_dict()}")
    print(f"  Kruskal-Wallis H={h:.2f}, p={p:.2e}, 유의(p<0.05)={p < 0.05}\n")


[월별 변동계수(CV)] 세그먼트별 평균: {'생활밀착형': 0.08, '로컬미식형': 0.1, '프리미엄외식형': 0.08}
  Kruskal-Wallis H=0.13, p=9.37e-01, 유의(p<0.05)=False

[건당 평균결제금액] 세그먼트별 평균: {'생활밀착형': 18276.96, '로컬미식형': 18402.23, '프리미엄외식형': 14640.12}
  Kruskal-Wallis H=52.79, p=3.45e-12, 유의(p<0.05)=True



## 5-4. ⭐ 특이 시그널: 50대·60대 이상 외국인 소비층

§3-3에서 "연령대별 소비 규모는 30대>40대>50대>20대>60대이상>20대이하 순이며,
50대·60대이상 비중이 낮지 않다"는 점이 지적되었다. 전수 데이터로 재확인한다.

In [5]:
age_total = foreign_df.groupby('AGE_LABEL')['amt'].sum().reindex(AGE_LABEL_LIST)
age_share = (age_total / age_total.sum() * 100).round(1)
print("연령대별 소비금액 및 비중 (전체 외국인, 6개월 합산):")
for age in AGE_LABEL_LIST:
    print(f"  {age}: {age_total[age]/1e8:,.0f}억원 ({age_share[age]}%)")

senior_share = age_share['50대'] + age_share['60대 이상']
print(f"\n50대+60대이상 합산 비중: {senior_share:.1f}% (참고: 20대 비중 {age_share['20대']}%)")
print("[해석] 50대 이상 비중이 20대와 비슷하거나 더 커서, '관광=젊은층' 이라는 통념과 다른 시그널이 확인된다.")


연령대별 소비금액 및 비중 (전체 외국인, 6개월 합산):
  20대 이하: 221억원 (1.7%)
  20대: 2,347억원 (18.4%)
  30대: 3,070억원 (24.0%)
  40대: 2,693억원 (21.1%)
  50대: 2,465억원 (19.3%)
  60대 이상: 1,985억원 (15.5%)

50대+60대이상 합산 비중: 34.8% (참고: 20대 비중 18.4%)
[해석] 50대 이상 비중이 20대와 비슷하거나 더 커서, '관광=젊은층' 이라는 통념과 다른 시그널이 확인된다.


In [6]:
# 세그먼트별 50대+60대이상 비중 비교
senior_by_seg = (merged.groupby('segment')['age_pct_50대'].mean() + merged.groupby('segment')['age_pct_60대 이상'].mean())
senior_by_seg = senior_by_seg.reindex(SEGMENT_ORDER)
print("세그먼트별 50대+60대이상 평균 비중(%):")
print(senior_by_seg.round(1))


세그먼트별 50대+60대이상 평균 비중(%):
segment
생활밀착형      26.5
로컬미식형      45.4
프리미엄외식형    31.4
dtype: float64


## 5-5. 이상치(outlier) 지역 케이스 스터디

전수 데이터에서 업종 비중이 세그먼트 평균과 크게 벗어나는 지역 2곳을 재확인한다.

In [7]:
print("=== 케이스 1: 영등포구 — 중국음식 비중 이례적 편중 ===")
yd = merged[merged['CCG_NM'] == '영등포구'][['SIDO_NM', 'CCG_NM', 'segment', 'buz_pct_중국음식', '총건수', '총소비금액']]
print(yd.to_string(index=False))
seg_avg_cn = merged.groupby('segment')['buz_pct_중국음식'].mean()
print(f"\n로컬미식형 평균 중국음식 비중: {seg_avg_cn['로컬미식형']:.1f}% -> 영등포구는 {yd['buz_pct_중국음식'].values[0]:.1f}%로 약 {yd['buz_pct_중국음식'].values[0]/seg_avg_cn['로컬미식형']:.1f}배")


=== 케이스 1: 영등포구 — 중국음식 비중 이례적 편중 ===
SIDO_NM CCG_NM segment  buz_pct_중국음식    총건수       총소비금액
  서울특별시   영등포구   로컬미식형     27.941517 969627 18124320000

로컬미식형 평균 중국음식 비중: 3.6% -> 영등포구는 27.9%로 약 7.7배


In [8]:
print("=== 케이스 2: 과천시 — 일반한식 비중 이례적 편중 ===")
gc = merged[merged['CCG_NM'] == '과천시'][['SIDO_NM', 'CCG_NM', 'segment', 'buz_pct_일반한식', '총건수', '총소비금액']]
print(gc.to_string(index=False))
seg_avg_hs = merged.groupby('segment')['buz_pct_일반한식'].mean()
print(f"\n로컬미식형 평균 일반한식 비중: {seg_avg_hs['로컬미식형']:.1f}% -> 과천시는 {gc['buz_pct_일반한식'].values[0]:.1f}%")
print("[해석] 과천시·영등포구 모두 '로컬미식형'으로 분류되지만, 세그먼트 평균을 훨씬 넘는 특정 업종 쏠림이 있어")
print("전국 단위 전략보다 지역 맞춤형 접근이 더 유효할 수 있는 대표 사례.")


=== 케이스 2: 과천시 — 일반한식 비중 이례적 편중 ===
SIDO_NM CCG_NM segment  buz_pct_일반한식   총건수     총소비금액
    경기도    과천시   로컬미식형     50.707868 56267 653370000

로컬미식형 평균 일반한식 비중: 32.1% -> 과천시는 50.7%
[해석] 과천시·영등포구 모두 '로컬미식형'으로 분류되지만, 세그먼트 평균을 훨씬 넘는 특정 업종 쏠림이 있어
전국 단위 전략보다 지역 맞춤형 접근이 더 유효할 수 있는 대표 사례.


## 요약

- 업종 구성비·연령 구성비·월별 CV·건당 결제금액 모두 세그먼트 간 통계적으로 유의한 차이(p<0.05)를 보임.
- 50대 이상(50대+60대이상) 외국인 소비 비중이 낮지 않다는 시그널을 전수 데이터로 재확인.
- 영등포구(중국음식)·과천시(일반한식) 사례는 세그먼트 평균으로 설명되지 않는 국지적 편중을 보여줌.

다음 단계(`06_visualization.ipynb`)에서 지금까지의 분석 결과를 시각화한다.